# Real VS Code sessions: private review and evaluation

Work from a **copy of this notebook under `data/private/`**, using the project `.venv` kernel. Do not save private widget state or executed outputs into the shareable notebook template. The prepared snapshot covers the five available sessions created in the selected 30-day window: 24 indexed user turns, including 9 routing-development turns that cannot be exported for evaluation.

The 15 other turns are candidates, not a benchmark. Many involve research, data work, publishing or follow-up approvals rather than the six coding roles. Do not force them into a coding category: review for fallback, ambiguity, missing context or exclusion. Keep prompts fixed while labeling and do not use model predictions as gold labels.

Only exact host-indexed user messages and earlier final replies are included. System/developer instructions, tool output, reasoning, delegated-agent requests and the current turn's later answer are excluded. Attachment contents are not imported. Redaction is heuristic: inspect every included query and context before approving remote use. This notebook makes no remote calls unless you explicitly enable evaluation after review.

In [ ]:
from pathlib import Path
from collections import Counter
from uuid import uuid4
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'contracts' / 'agents.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook inside the routing experiment project')
sys.path.insert(0, str(ROOT))
from routing_lab.data import ROUTES, load_contracts
from routing_lab.session_data import ReviewCandidate, load_review_queue, export_reviewed, load_approved_records, input_digest
from routing_lab.endpoint import ClefEndpointBackend
from routing_lab.evaluation import run_experiment
from routing_lab.prompt_experiment import expanded_contracts

PRIVATE_DIR = ROOT / 'data' / 'private' / 'session-review-20260930-103807-v2'
review_pointer = PRIVATE_DIR / 'active-review.json'
REVIEW_QUEUE_PATH = PRIVATE_DIR / 'review-queue.jsonl'
if review_pointer.exists():
    review_name = json.loads(review_pointer.read_text(encoding='utf-8'))['file']
    if Path(review_name).name != review_name or not review_name.startswith('reviewed-'):
        raise ValueError('Invalid local review pointer')
    REVIEW_QUEUE_PATH = PRIVATE_DIR / review_name
queue = load_review_queue(REVIEW_QUEUE_PATH)
print('Loaded private review snapshot:', REVIEW_QUEUE_PATH)
print('Total:', len(queue), '| Development-only:', sum(row.development_only for row in queue))
print('Included:', sum(row.include for row in queue), '| Explicit remote approval:', sum(row.remote_approved for row in queue))
print('Review flags:', dict(Counter(flag for row in queue for flag in row.flags)))

## Review one decision at a time

Choose a turn, inspect the redacted query and prior context, and open its source session if needed. Restore only information that was available **before** that request was routed, for example the original attached code snapshot. Do not copy the assistant's eventual diagnosis or generated solution into that turn's input.

Select the initial owning role and any genuinely acceptable alternatives. `Include` means use this example for a reviewed pilot; `Approve ... Azure` explicitly approves the exact edited text for transmission to your deployed endpoint. Save each decision before switching turns. All fields default to unreviewed/unapproved. Development-only turns remain visible but cannot be included.

Saving creates a new private review snapshot and a local pointer for resuming after restart. It does not modify the source chat or the original extraction. An excluded row needs no invented label.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

selector = widgets.Dropdown(options=[(f'{i + 1}: {row.id}' + (' [DEV ONLY]' if row.development_only else ''), i) for i, row in enumerate(queue)], description='Turn:', layout=widgets.Layout(width='95%'))
details = widgets.Output()
query_editor = widgets.Textarea(description='Query:', layout=widgets.Layout(width='95%', height='130px'))
context_editor = widgets.Textarea(description='Context:', layout=widgets.Layout(width='95%', height='220px'))
label = widgets.Dropdown(options=[('Unlabeled', None)] + [(route, route) for route in ROUTES], description='Owner:')
acceptable = widgets.SelectMultiple(options=ROUTES, description='Acceptable:', rows=6)
include = widgets.Checkbox(description='Include in reviewed evaluation', value=False)
context_checked = widgets.Checkbox(description='Reviewed context completeness and timing', value=False, indent=False)
remote_ok = widgets.Checkbox(description='Approve this edited input for the Azure Clef endpoint', value=False, indent=False)
reviewer = widgets.Text(description='Reviewer:', placeholder='Your name')
notes = widgets.Textarea(description='Notes:', layout=widgets.Layout(width='95%'))
save = widgets.Button(description='Save review snapshot', button_style='primary')
status = widgets.Output()
loading_selection = False

def load_selection(change=None):
    global loading_selection
    loading_selection = True
    row = queue[selector.value]
    query_editor.value, context_editor.value = row.query, row.context
    label.value, acceptable.value = row.expected_route, tuple(row.acceptable_routes)
    include.value, include.disabled = row.include, row.development_only
    context_checked.value, remote_ok.value = row.context_reviewed, row.remote_approved
    reviewer.value, notes.value = row.reviewer, row.notes
    loading_selection = False
    with details:
        details.clear_output()
        display(Markdown(f'[Open source session](agent-host-session://copilotcli/{row.session_id})'))
        print('Time:', row.timestamp, '| Group:', row.group_id, '| Workspace:', row.repository)
        print('Flags:', ', '.join(row.flags) or 'None detected; manual privacy review still required')

def save_selection(button):
    global REVIEW_QUEUE_PATH
    with status:
        status.clear_output()
        if include.value and (label.value is None or not reviewer.value.strip() or not context_checked.value or not remote_ok.value):
            print('Not saved: included rows need a label, reviewer, context check and explicit Azure approval.')
            return
        if not query_editor.value.strip():
            print('Not saved: query must not be empty.')
            return
        row = queue[selector.value]
        if include.value and row.development_only:
            raise ValueError('Routing-development turns cannot enter evaluation')
        alternatives = list(acceptable.value)
        if label.value is not None and label.value not in alternatives:
            alternatives.append(label.value)
        reviewed = ReviewCandidate.model_validate({**row.model_dump(),
            'query': query_editor.value, 'context': context_editor.value,
            'include': include.value, 'expected_route': label.value, 'acceptable_routes': alternatives,
            'review_status': 'reviewed' if include.value else 'unreviewed',
            'context_reviewed': context_checked.value, 'remote_approved': remote_ok.value,
            'approved_input_sha256': input_digest(query_editor.value, context_editor.value) if context_checked.value and remote_ok.value else '',
            'reviewer': reviewer.value.strip(), 'notes': notes.value})
        updated_queue = list(queue)
        updated_queue[selector.value] = reviewed
        snapshot = PRIVATE_DIR / f'reviewed-{uuid4().hex[:10]}.jsonl'
        with snapshot.open('x', encoding='utf-8') as stream:
            for item in updated_queue:
                stream.write(item.model_dump_json() + '\n')
        review_pointer.write_text(json.dumps({'file': snapshot.name}), encoding='utf-8')
        queue[selector.value] = reviewed
        REVIEW_QUEUE_PATH = snapshot
        print('Saved:', snapshot)
        print('Included so far:', sum(item.include for item in queue))

def invalidate_input_approval(change):
    if not loading_selection:
        context_checked.value = False
        remote_ok.value = False

query_editor.observe(invalidate_input_approval, names='value')
context_editor.observe(invalidate_input_approval, names='value')
selector.observe(load_selection, names='value')
save.on_click(save_selection)
load_selection()
display(widgets.VBox([selector, details, query_editor, context_editor, label, acceptable, include, context_checked, remote_ok, reviewer, notes, save, status]))

## Export the approved subset

Export refuses incomplete approvals, missing labels or included development-only turns. Records keep a common session group and workspace group, so the existing split utilities keep related turns together. Two chats from the same workspace do not become independent train/test groups. With at most 15 candidates across three non-development workspaces, this is a small pilot, not a balanced six-route benchmark.

The exported dataset has a hash-bound approval manifest. Editing it afterward invalidates approval until it is reviewed and exported again.

In [ ]:
EXPORT_REVIEWED = False
APPROVED_DATA_DIR = None  # Set to a previously exported folder when resuming.
if EXPORT_REVIEWED:
    APPROVED_DATA_DIR = ROOT / 'data' / 'reviewed' / f'real-sessions-{uuid4().hex[:10]}'
    approval = export_reviewed(REVIEW_QUEUE_PATH, APPROVED_DATA_DIR)
    print('Export:', APPROVED_DATA_DIR)
    print(json.dumps(approval, indent=2))
else:
    print('No export. Review and save candidates, then enable EXPORT_REVIEWED.')

## Evaluate approved real inputs on Azure

Select `baseline` or `expanded_v1`; neither changes model weights. Freeze the criteria before evaluating. Repeatedly optimizing on these same reviewed examples turns them into development data, not a held-out test. Keep the two prompt versions and all results; do not overwrite a poor result.

Both switches below must be explicitly enabled. The approval hash is checked before constructing the endpoint client. Only the reviewed query/context and routing criteria are sent, never the expected labels, notes or reviewer identity. Do not run this step until you have permission to send every included input to Azure.

In [ ]:
RUN_REAL_EVALUATION = False
ALLOW_PRIVATE_REMOTE = False
REAL_VARIANT = 'baseline'
if RUN_REAL_EVALUATION:
    if not ALLOW_PRIVATE_REMOTE or APPROVED_DATA_DIR is None:
        raise RuntimeError('Export reviewed data and explicitly approve remote evaluation first')
    if REAL_VARIANT not in {'baseline', 'expanded_v1'}:
        raise ValueError('Choose baseline or expanded_v1')
    real_records = load_approved_records(Path(APPROVED_DATA_DIR))
    real_contracts = load_contracts(ROOT / 'contracts' / 'agents.json')
    if REAL_VARIANT == 'expanded_v1':
        real_contracts = expanded_contracts(real_contracts, ROOT / 'contracts' / 'criteria-expanded-v1.json')
    real_connection = json.loads((ROOT / 'azure' / 'online' / 'connection.json').read_text())
    real_backend = ClefEndpointBackend(real_contracts, url=real_connection['scoring_uri'], context_mode='context')
    real_output = ROOT / 'outputs' / f'real-session-pilot-{REAL_VARIANT}-{uuid4().hex[:10]}'
    real_summary = run_experiment(real_backend, real_records, real_output, probe=False,
        source_paths={'dataset': Path(APPROVED_DATA_DIR) / 'records.jsonl', 'approval': Path(APPROVED_DATA_DIR) / 'approval.json',
                      'contracts': ROOT / 'contracts' / 'agents.json', 'expanded_criteria': ROOT / 'contracts' / 'criteria-expanded-v1.json'})
    print('Small reviewed pilot, not production validation:', real_output)
    print(json.dumps(real_summary, indent=2))
else:
    print('No private session data sent to Azure. RUN_REAL_EVALUATION remains disabled.')